# 04-08 - Data Preparation Synthesis

**Phase:** 04 - Data Analysis & Preparation

**Difficulty:** 3/3 | **Priority:** 2/2

**Status:** VERIFIED

---

## 1. What Are We Solving?

This is the **capstone** for Phase 04. We bring together everything: EDA, cleaning, missing values, outliers, scaling, encoding, splits, and leakage prevention - into one complete data pipeline.

## 2. Why Does This Matter?

Real ML projects require a full data preparation pipeline. This notebook shows how all the pieces fit together to produce clean, model-ready data without leakage.

## 3. Prerequisites

- All of Phase 04 (Units 04.1-04.7)
- Phase 03 (Statistics)

## 4. Learning Objectives

By the end of this notebook, you should be able to:
- [ ] Build a complete data preparation pipeline
- [ ] Handle missing values, outliers, and encoding
- [ ] Split data correctly without leakage
- [ ] Use scikit-learn pipelines
- [ ] Evaluate a model on properly prepared data
## 5. Mental Model

**Mental Model:** This is your final exam for data preparation. Like a chef who has learned to chop, sauté, season, and plate individually, you now need to combine all these skills into one coherent dish. The pipeline is your recipe — it ensures every step happens in the right order, every time, for every piece of data.

Key: understand the data before you model it.



## 2a. Decision Guidance

**Decision Guidance:**

| Situation | What to Do | Why |
|-----------|------------|-----|
| Multiple data quality issues | Build a Pipeline that handles all in order | Consistency and reproducibility |
| Mixed numeric and categorical | Use ColumnTransformer | Different transforms for different types |
| Need to evaluate different strategies | Use cross-validation with Pipeline | Fair comparison without leakage |
| Deploying to production | Save the entire Pipeline | Reproducibility and no leakage |
| Team collaboration | Document all preprocessing decisions | Reproducibility and debugging |


## 2b. Common Mistakes to Avoid

**Common Mistakes to Avoid:**
- Not using Pipeline (manual steps get out of order)
- Fitting preprocessing outside cross-validation
- Not saving the fitted pipeline for production
- Not documenting preprocessing decisions
- Skipping the validation set and testing directly


## 6. Create a Messy Dataset

Let's create a realistic messy dataset with missing values, outliers, categorical data, and inconsistent formats.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)
n = 1000

df = pd.DataFrame({
    "age": np.random.randint(18, 70, n).astype(float),
    "income": np.random.normal(60000, 20000, n),
    "spend": np.random.normal(3000, 1000, n),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "churned": np.random.choice([0, 1], n, p=[0.7, 0.3]),
})

# Add messiness: missing values, outliers, inconsistent region
df.loc[np.random.choice(n, 80, replace=False), "income"] = np.nan
df.loc[np.random.choice(n, 50, replace=False), "age"] = np.nan
df.loc[np.random.choice(n, 5, replace=False), "spend"] = 50000  # outliers
df.loc[np.random.choice(n, 30, replace=False), "region"] = "north"  # inconsistent case

print("Messy dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nRegion values (inconsistent case):")
print(df["region"].value_counts())


Messy dataset shape: (1000, 5)

Missing values:
age        50
income     80
spend       0
region      0
churned     0
dtype: int64

Region values (inconsistent case):
region
West     248
East     248
South    239
North    235
north     30
Name: count, dtype: int64


## 7. Step 1: Clean

Standardize the region column and check for duplicates.


In [2]:
# Clean: standardize region case
df["region"] = df["region"].str.strip().str.title()
print("Region after standardizing:")
print(df["region"].value_counts())

# Check duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"After removing duplicates: {len(df)} rows")


Region after standardizing:
region
North    265
West     248
East     248
South    239
Name: count, dtype: int64

Duplicate rows: 0
After removing duplicates: 1000 rows


## 8. Step 2: Handle Outliers

Cap extreme spend values using the IQR method.


In [3]:
# Handle outliers: cap spend at IQR bounds
q1 = df["spend"].quantile(0.25)
q3 = df["spend"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

n_outliers = ((df["spend"] > upper) | (df["spend"] < lower)).sum()
df["spend"] = df["spend"].clip(lower, upper)

print(f"Spend bounds: [{lower:.0f}, {upper:.0f}]")
print(f"Outliers capped: {n_outliers}")
print(f"Max spend after capping: {df['spend'].max():.0f}")


Spend bounds: [391, 5676]
Outliers capped: 14
Max spend after capping: 5676


## 9. Step 3: Split (Before Preprocessing!)

Critical: split into train/val/test BEFORE any preprocessing to prevent leakage.


In [4]:
# Split first to prevent leakage
X = df.drop("churned", axis=1)
y = df["churned"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"Class balance preserved: train={y_train.mean():.2f}, val={y_val.mean():.2f}, test={y_test.mean():.2f}")
print("\nWe split BEFORE preprocessing to avoid leakage.")


Train: 700, Val: 150, Test: 150
Class balance preserved: train=0.27, val=0.27, test=0.27

We split BEFORE preprocessing to avoid leakage.


## 10. Step 4: Build a Preprocessing Pipeline

Use a `ColumnTransformer` to apply different preprocessing to numeric and categorical columns, then chain with a model in a pipeline.


In [5]:
# Define preprocessing for numeric and categorical columns
numeric_cols = ["age", "income", "spend"]
categorical_cols = ["region"]

numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

# Full pipeline: preprocess + model
pipeline = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
print("Pipeline built:")
print(pipeline)


Pipeline built:
Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  ['age', 'income', 'spend']),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
       

## 11. Step 5: Train and Validate

Fit the pipeline on training data and evaluate on validation.


In [6]:
# Train on train, validate on val
pipeline.fit(X_train, y_train)
val_acc = accuracy_score(y_val, pipeline.predict(X_val))
train_acc = accuracy_score(y_train, pipeline.predict(X_train))

print(f"Train accuracy: {train_acc:.3f}")
print(f"Validation accuracy: {val_acc:.3f}")
print("\nThe pipeline handles imputation, scaling, and encoding automatically.")


Train accuracy: 0.727
Validation accuracy: 0.727

The pipeline handles imputation, scaling, and encoding automatically.


## 12. Step 6: Final Evaluation on Test

Evaluate once on the held-out test set.


In [7]:
# Final evaluation on test (used once)
test_acc = accuracy_score(y_test, pipeline.predict(X_test))
print(f"Final test accuracy: {test_acc:.3f}")
print("\nThe test set was never used for training or tuning.")


Final test accuracy: 0.727

The test set was never used for training or tuning.


## 13. Step 7: Cross-Validation on Train

For a more reliable estimate, use cross-validation on the training set.


In [8]:
from sklearn.model_selection import cross_val_score

# Cross-validate the full pipeline on training data
scores = cross_val_score(pipeline, X_train, y_train, cv=5)
print(f"5-fold CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")
print("\nThe pipeline prevents leakage during cross-validation.")


5-fold CV accuracy: 0.727 +/- 0.003

The pipeline prevents leakage during cross-validation.


## 14. Failure Case: Leaky Pipeline

If we impute/scale before splitting, we leak test information. Let's demonstrate the difference.


In [9]:
# LEAKY version: preprocess all data before splitting
X_all = df.drop("churned", axis=1)
y_all = df["churned"]

# Fit preprocessor on ALL data (leakage)
preprocessor_leak = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical_cols),
])
X_all_prep = preprocessor_leak.fit_transform(X_all)

X_tr, X_te, y_tr, y_te = train_test_split(X_all_prep, y_all, test_size=0.2, random_state=42, stratify=y_all)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
leak_acc = accuracy_score(y_te, model.predict(X_te))

print(f"Accuracy with leaky preprocessing: {leak_acc:.3f}")
print(f"Accuracy with correct pipeline: {test_acc:.3f}")
print("\nThe leaky version uses test information in preprocessing.")
print("Always split before fitting any transformer.")


Accuracy with leaky preprocessing: 0.725
Accuracy with correct pipeline: 0.727

The leaky version uses test information in preprocessing.
Always split before fitting any transformer.


## 15. Debugging: Common Errors

- **Preprocessing before splitting**: leakage.
- **Not handling missing values**: models fail.
- **Not encoding categoricals**: models fail.
- **Not scaling**: distance models underperform.
- **Tuning on test**: overestimates performance.

## 16. Real-World Considerations

- Use pipelines for consistency and to prevent leakage.
- Split before any preprocessing.
- Validate on a separate set, test once at the end.
- Document your preprocessing steps.

## 17. Common Mistakes

- Fitting transformers on all data.
- Forgetting to encode categoricals.
- Using the test set for tuning.
- Not checking for leakage.

## 18. When NOT to Use

- Don't preprocess before splitting.
- Don't use the test set more than once.
- Don't skip EDA before building the pipeline.

## 19. Challenge

Add a new categorical feature to the pipeline and verify the full pipeline still works end-to-end.


In [10]:
# Challenge: add a categorical feature and rebuild pipeline
df["plan"] = np.random.choice(["basic", "premium", "enterprise"], len(df))

X2 = df.drop("churned", axis=1)
y2 = df["churned"]

X2_train, X2_temp, y2_train, y2_temp = train_test_split(X2, y2, test_size=0.3, random_state=42, stratify=y2)
X2_val, X2_test, y2_val, y2_test = train_test_split(X2_temp, y2_temp, test_size=0.5, random_state=42, stratify=y2_temp)

cat_cols2 = ["region", "plan"]
preprocessor2 = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat_cols2),
])
pipeline2 = make_pipeline(preprocessor2, LogisticRegression(max_iter=1000))
pipeline2.fit(X2_train, y2_train)
acc2 = accuracy_score(y2_test, pipeline2.predict(X2_test))

print(f"Test accuracy with 'plan' feature: {acc2:.3f}")
print("\nThe pipeline handles the new categorical feature automatically.")


Test accuracy with 'plan' feature: 0.727

The pipeline handles the new categorical feature automatically.


## 20. Closed-Book Recall

Without looking back:

1. What are the steps in a complete data pipeline?
2. Why split before preprocessing?
3. What does a ColumnTransformer do?
4. Why use a pipeline?
5. How do you prevent leakage in cross-validation?

## 21. Teach-Back Questions

Explain to another person:

- How to build a complete data preparation pipeline.
- Why pipelines prevent leakage.
- The importance of splitting before preprocessing.

## 22. Summary

You've completed Phase 04! You can now explore, clean, and prepare data end-to-end: handle missing values and outliers, encode and scale features, split correctly, and prevent leakage with pipelines. This is the foundation for building reliable ML models.

## 23. Further Experiment

- Add feature engineering (e.g. income per person).
- Use a tree-based model that handles missing values natively.
- Build a reusable data pipeline function.

## 24. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, pandas, matplotlib, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
